# Roketsan araç tespiti — final ensemble

Bu notebook, gece farklı makinelerde (EVREN, Colab, Mac) yaptığımız işin tek dosyadaki özetidir.
Ağırlıklar zip içinde, Kaggle'a gönderilen dosyalar `submissions/` klasöründe.

| Kurgu | Detektörler | Val (809) | Kaggle public |
|---|---|---|---|
| **final5** (ana) | yolo26m_best, D2, colab26m, C, copypaste_van | 0,865 | **0,801** |
| final3 (yedek) | yolo26m_best, D2, colab26m | 0,850 | 0,7988 |

**Boru hattı**
1. Her model: tam görüntü + 800 px tile (SAHI), conf 0,001
2. TTA: yatay çevirme ve 1600 px (copypaste_van hariç)
3. Ağırlıklı WBF: SAHI üyeleri 1, TTA üyeleri 0,5
4. Van/car sınıflandırıcısı: skor = 0,7·detektör + 0,3·sınıflandırıcı
5. 150 px²'den küçük kutular atılır, görüntü başına en fazla 500 kutu

Bölümler: **A · Eğitim** (veri ve eğitim tarifleri) · **B · Çıkarım** (ensemble ve submission)

In [ ]:
!pip -q install ultralytics timm ensemble-boxes

## Ayarlar

In [ ]:
from pathlib import Path

DATA = Path("data")              # Kaggle verisi: data/train/{images,annotations.csv}, data/test/images, data/sample_submission.csv
WEIGHTS = Path(".")              # ağırlıkların kök klasörü (paketin açıldığı yer)
SPLIT = "test"                   # "test" → submission · "val" → val_ids.txt üzerinde mAP@0.5
ENSEMBLE = "final5"              # "final5" → ana submission (Kaggle 0,801) · "final3" → güvenlik (Kaggle 0,7988)
VAL_IDS_FILE = WEIGHTS / "val_ids.txt"
CACHE = Path("cache_final")      # ara tahminler
OUT_CSV = Path("submission.csv")

DETECTORS = {                    # ad → ağırlık
    "yolo26m_best": WEIGHTS / "yolo26m_best.pt",
    "D2": WEIGHTS / "outputs/d2/D2.pt",
    "colab26m": WEIGHTS / "outputs/colab26m/colab26m.pt",
}
if ENSEMBLE == "final5":
    DETECTORS["C"] = WEIGHTS / "outputs/evren/C.pt"
    DETECTORS["copypaste_van"] = WEIGHTS / "outputs/copypaste_van/copypaste_van.pt"
NO_TTA = {"copypaste_van"}       # bu modellerin yalnızca SAHI üyesi var
CLASSIFIER = WEIGHTS / "outputs/classifier/cls_convnext.pt"

CLASSES = ["car", "van", "truck", "bus"]
CONF, MAX_DET_MODEL, MIN_AREA_PRED = 0.001, 300, 150
TILE, TILE_OVERLAP, TILE_IMGSZ = 800, 0.2, 1280
SAHI_IOU, MAX_DET = 0.6, 500
WBF_IOU, WBF_TOP = 0.6, 300
TTA_WEIGHT = 0.5
CLS_A, CLS_MIN_SCORE = 0.3, 0.01

In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from torchvision.ops import batched_nms

DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
TORCH_DEVICE = "cuda" if DEVICE == 0 else DEVICE
CID = {c: i for i, c in enumerate(CLASSES)}
CACHE.mkdir(exist_ok=True)

if SPLIT == "test":
    IMG_DIR = DATA / "test" / "images"
    IDS = pd.read_csv(DATA / "sample_submission.csv").image_id.tolist()
else:
    IMG_DIR = DATA / "train" / "images"
    IDS = VAL_IDS_FILE.read_text().split()
SIZES = {i: Image.open(IMG_DIR / f"{i}.jpg").size for i in IDS}
print(f"{SPLIT}: {len(IDS)} görüntü · cihaz {DEVICE}")

# A · Eğitim

Tarifler checkpoint'lerde kayıtlı ayarlardan (`train_args`) okundu; yalnızca varsayılandan farklı olanlar yazıldı.

| Model | Veri | Donanım | Not |
|---|---|---|---|
| D2 · YOLO26l | v2.0 | EVREN, 4× H200 | 60 epoch, 1 sa 11 dk |
| colab26m · YOLO26m | baseline | Colab, A100 | en iyi epoch 45 |
| yolo26m_best · YOLO26m | ekip modeli | EVREN | 200 epoch |
| C · YOLO26m | v2.0 | EVREN, 2× H200 | en iyi epoch 42 |
| copypaste_van · YOLO26m | ekibin copy-paste seti | EVREN | 60 epoch |
| Sınıflandırıcı · ConvNeXt-Tiny | kutu kırpıntıları | Mac M4 Pro | 8 epoch |

Ensemble'ın kazancı, modellerin farklı tariflerle eğitilip farklı hatalar yapmasından geliyor. final3 modellerinin ana farkları:

| | D2 | colab26m | yolo26m_best |
|---|---|---|---|
| Başlangıç | yolo26l | yolo26m | yolo26m |
| Veri | v2.0 | baseline | baseline (varsayım) |
| Optimizer · lr0 | AdamW · 0,002 | MuSGD · 0,00038 | AdamW · 0,002 |
| Batch | 64 | 16 | 304 |
| mixup | 0,1 | 0,05 | 0,1 |
| flipud · scale · döndürme | 0 · 0,5 · 0° | 0 · 0,5 · 0° | 0,5 · 0,9 · 5° |
| Val (SAHI 800) | 0,794 | 0,794 | 0,790 |

In [ ]:
TRAIN_DATA = False        # A1 · YOLO veri setlerini hazırla (baseline, v2.0)
TRAIN_D2 = False          # A2 · YOLO26l
TRAIN_COLAB26M = False    # A3 · YOLO26m, MuSGD
TRAIN_TEAM26M = False     # A4 · ekip modelinin tarifi
TRAIN_C = False           # A5 · C — YOLO26m, v2.0 (final5)
TRAIN_COPYPASTE_VAN = False  # A6 · copypaste_van — ekibin copy-paste veri seti (final5)
TRAIN_CLASSIFIER = False  # A7 · van/car sınıflandırıcısı
YOLO_DATA = Path("yolo_data")          # hazırlanan veri setleri
RUNS = Path("train_runs")              # eğitim çıktıları; bitince DETECTORS / CLASSIFIER yollarını buraya çevirin

## A1 · Veri setleri
- **baseline:** 5.662 train görüntüsü, olduğu gibi.
- **v2.0:** baseline + gece sahnelerine (`night_scene_ids.txt`) 3, bus içeren görüntülere 1 ek kopya (yatay çevrilmiş)
  + büyük görüntülerden 1.280 px tile'lar. Toplam 12.632 dosya.

Val iki sette de aynı 809 görüntü (`val_ids.txt`).

In [ ]:
import os
import shutil
from concurrent.futures import ThreadPoolExecutor

MIN_AREA_LABEL = 200.0
TILE_SIZE_TRAIN, TILE_MIN_SIDE, TILE_QUALITY = 1280, 1400, 80


def build_datasets():
    ann = pd.read_csv(DATA / "train" / "annotations.csv"); ann["label"] = ann.label.str.strip().str.lower()
    boxes = {k: g[["x", "y", "w", "h", "label"]].to_numpy() for k, g in ann.groupby("image_id")}
    all_train = sorted(p.stem for p in (DATA / "train" / "images").glob("*.jpg"))
    val_ids = set(VAL_IDS_FILE.read_text().split())
    train_ids = [i for i in all_train if i not in val_ids]
    night = set((WEIGHTS / "night_scene_ids.txt").read_text().split()) & set(train_ids)
    bus = set(ann.loc[ann.label == "bus", "image_id"]) & set(train_ids)

    def label(b, W, H):
        return "".join(f"{CID[l]} {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}\n" for x, y, w, h, l in b)

    def hflip(b, W):
        b = b.copy(); b[:, 0] = W - b[:, 0].astype(float) - b[:, 2].astype(float); return b

    def clip(b, win, min_visible=0.6):
        if len(b) == 0:
            return b
        x0, y0, x1, y1 = win; f = b[:, :4].astype(float)
        bx1, by1 = np.maximum(f[:, 0], x0), np.maximum(f[:, 1], y0)
        bx2, by2 = np.minimum(f[:, 0] + f[:, 2], x1), np.minimum(f[:, 1] + f[:, 3], y1)
        cw, ch = np.clip(bx2 - bx1, 0, None), np.clip(by2 - by1, 0, None)
        keep = (cw * ch >= min_visible * f[:, 2] * f[:, 3]) & (cw * ch >= MIN_AREA_LABEL)
        out = np.empty((keep.sum(), 5), dtype=object)
        out[:, 0], out[:, 1], out[:, 2], out[:, 3], out[:, 4] = bx1[keep] - x0, by1[keep] - y0, cw[keep], ch[keep], b[keep, 4]
        return out

    def write(args):
        image_id, split_dir, mode = args
        src = DATA / "train" / "images" / f"{image_id}.jpg"
        b = boxes.get(image_id, np.empty((0, 5), dtype=object))
        idir, ldir = split_dir / "images", split_dir / "labels"
        if mode == "copy":
            W, H = Image.open(src).size
            try:
                os.link(src, idir / src.name)
            except OSError:
                shutil.copy(src, idir / src.name)
            (ldir / f"{image_id}.txt").write_text(label(b, W, H)); return
        im = Image.open(src).convert("RGB"); W, H = im.size
        if mode.startswith("dup"):
            k = mode[3:]
            ImageOps.mirror(im).save(idir / f"{image_id}_dup{k}.jpg", quality=95 - int(k))
            (ldir / f"{image_id}_dup{k}.txt").write_text(label(hflip(b, W), W, H)); return
        if mode == "tile" and max(W, H) > TILE_MIN_SIDE:
            for j, win in enumerate(windows(W, H, TILE_SIZE_TRAIN, 0.2)):
                if win[2] - win[0] == W and win[3] - win[1] == H:
                    continue
                tb = clip(b, win)
                if len(tb) == 0 and j % 5:
                    continue
                im.crop(win).save(idir / f"{image_id}_t{j}.jpg", quality=TILE_QUALITY)
                (ldir / f"{image_id}_t{j}.txt").write_text(label(tb, win[2] - win[0], win[3] - win[1]))

    out = {}
    for name in ("baseline", "v2"):
        d = YOLO_DATA / name
        if (d / "data.yaml").exists():
            out[name] = d / "data.yaml"; continue
        for sp in ("train", "valid"):
            (d / sp / "images").mkdir(parents=True, exist_ok=True); (d / sp / "labels").mkdir(parents=True, exist_ok=True)
        jobs = [(i, d / "valid", "copy") for i in sorted(val_ids)] + [(i, d / "train", "copy") for i in train_ids]
        if name == "v2":
            jobs += [(i, d / "train", f"dup{k}") for i in sorted(night) for k in (1, 2, 3)]
            jobs += [(i, d / "train", "dup1") for i in sorted(bus - night)]
            jobs += [(i, d / "train", "tile") for i in train_ids]
        with ThreadPoolExecutor(8) as ex:
            list(ex.map(write, jobs))
        (d / "data.yaml").write_text(f"path: {d.resolve()}\ntrain: train/images\nval: valid/images\nnc: 4\nnames: {CLASSES}\n")
        print(f"{name}: train {len(list((d / 'train' / 'images').iterdir()))} · val {len(list((d / 'valid' / 'images').iterdir()))}")
        out[name] = d / "data.yaml"
    return out


# windows() B bölümünde de kullanılıyor; burada tanımlı değilse aşağıdaki tanım geçerli
def windows(W, H, size=800, overlap=0.2):
    def st(n):
        if n <= size:
            return [0]
        step = int(size * (1 - overlap))
        return sorted(set(list(range(0, n - size, step)) + [n - size]))
    return [(x, y, min(x + size, W), min(y + size, H)) for y in st(H) for x in st(W)]


if TRAIN_DATA or TRAIN_D2 or TRAIN_COLAB26M or TRAIN_TEAM26M or TRAIN_C:
    DATASETS = build_datasets()

## A2 · D2 — YOLO26l, v2.0
EVREN'de 4 GPU (toplam batch 64).

In [ ]:
from ultralytics import YOLO

if TRAIN_D2:
    YOLO("yolo26l.pt").train(
        data=str(DATASETS["v2"]), imgsz=1280, epochs=60, batch=64, device=DEVICE,
        optimizer="AdamW", lr0=0.002, cos_lr=True, mixup=0.1,
        project=str(RUNS), name="D2", exist_ok=True,
    )  # diğer tüm ayarlar Ultralytics varsayılanı: lrf 0.01, warmup 3, mosaic 1, scale 0.5, fliplr 0.5, close_mosaic 10

## A3 · colab26m — YOLO26m, baseline, MuSGD
Resmi YOLO26 tarifi, 5,5 saatlik süre bütçesi. En iyi val (0,797) epoch 45'teydi; o checkpoint kullanıldı.

In [ ]:
if TRAIN_COLAB26M:
    YOLO("yolo26m.pt").train(
        data=str(DATASETS["baseline"]), imgsz=1280, time=5.5, epochs=1000, batch=16, seed=7, device=DEVICE,
        optimizer="MuSGD", lr0=0.00038, lrf=0.882, momentum=0.948, weight_decay=0.00027, warmup_epochs=1.0,
        cos_lr=True, box=8.0, cls=0.7, mixup=0.05, patience=30,
        project=str(RUNS), name="colab26m", exist_ok=True,
    )

## A4 · yolo26m_best — ekip modeli
Tarif checkpoint'ten okundu; veri seti kayıtlı olmadığı için baseline varsayıldı.

In [ ]:
if TRAIN_TEAM26M:
    YOLO("yolo26m.pt").train(
        data=str(DATASETS["baseline"]), imgsz=1280, epochs=200, batch=304, patience=80, device=DEVICE,
        optimizer="AdamW", lr0=0.002, lrf=0.001, weight_decay=0.001, warmup_epochs=5.0, warmup_bias_lr=0.0002,
        cos_lr=True, close_mosaic=15, degrees=5, scale=0.9, flipud=0.5, mixup=0.1,
        project=str(RUNS), name="team26m", exist_ok=True,
    )

## A5 · C — YOLO26m, v2.0
EVREN'de 2 GPU. En iyi val epoch 42'de (0,806); 77. epoch'ta erken durdu.

In [ ]:
if TRAIN_C:
    YOLO("yolo26m.pt").train(
        data=str(DATASETS["v2"]), imgsz=1280, epochs=120, batch=32, patience=30, device=DEVICE,
        optimizer="AdamW", lr0=0.005, lrf=0.001, weight_decay=0.001, warmup_epochs=5.0,
        cos_lr=True, close_mosaic=15, mixup=0.1,
        project=str(RUNS), name="C", exist_ok=True,
    )

## A6 · copypaste_van — YOLO26m, ekibin copy-paste veri seti
Ekip arkadaşımız EVREN'de eğitti (9.609 train dosyası, 971 val).
Test görüntülerini görmediği için submission'da kullanımı geçerli.

In [ ]:
if TRAIN_COPYPASTE_VAN:
    COPYPASTE_DATA = Path("drone_cp_final/data.yaml")   # ekibin veri seti (bu pakette yok)
    YOLO("yolo26m.pt").train(
        data=str(COPYPASTE_DATA), imgsz=1280, epochs=60, batch=64, device=DEVICE,
        optimizer="AdamW", lr0=0.002, warmup_bias_lr=0.0002, cos_lr=True,
        project=str(RUNS), name="copypaste_van", exist_ok=True,
    )

## A7 · Van/car sınıflandırıcısı — ConvNeXt-Tiny
- Girdi: kutunun 1,5 katı bağlamlı kare kırpıntı, 128 px; car örnekleri 80.000 ile sınırlı.
- Eğitim: RandomResizedCrop, çevirme, renk sarsıntısı · sınıf ağırlıklı CrossEntropy · AdamW, 8 epoch.
- Val doğruluğu: car 0,94 · van 0,74 · truck 0,89 · bus 0,88.

In [ ]:



def _crop_one(args):
    img_id, bxs, crop, context = args
    im = Image.open(DATA / "train" / "images" / f"{img_id}.jpg").convert("RGB"); out = []
    for x, y, w, h in bxs:
        cx, cy, s = x + w / 2, y + h / 2, max(w, h) * context
        out.append(np.asarray(im.crop((cx - s / 2, cy - s / 2, cx + s / 2, cy + s / 2)).resize((crop, crop), Image.BILINEAR)))
    return out


def train_classifier(epochs=8, bs=128, crop=144, context=1.5, inp=128, max_car=80_000, out=RUNS / "cls_convnext.pt"):
    import time
    import timm
    import torch.nn as nn
    import torchvision.transforms.v2 as T
    ann = pd.read_csv(DATA / "train" / "annotations.csv"); ann["label"] = ann.label.str.strip().str.lower()
    ann = ann[ann.w * ann.h >= 150]; val_ids = set(VAL_IDS_FILE.read_text().split())
    tr = ann[~ann.image_id.isin(val_ids)]; cars = tr[tr.label == "car"]
    tr = pd.concat([cars.sample(min(max_car, len(cars)), random_state=0), tr[tr.label != "car"]])
    va = ann[ann.image_id.isin(val_ids)]

    def crops(df):
        groups = list(df.groupby("image_id"))
        y = np.concatenate([g.label.map(CID).to_numpy() for _, g in groups])
        with ThreadPoolExecutor(8) as ex:
            X = [c for cs in ex.map(_crop_one, [(k, g[["x", "y", "w", "h"]].to_numpy(), crop, context) for k, g in groups], chunksize=16) for c in cs]
        return np.stack(X), y

    Xtr, ytr = crops(tr); Xva, yva = crops(va); print("kırpıntılar", Xtr.shape, Xva.shape)
    train_tf = T.Compose([T.RandomResizedCrop(inp, scale=(0.55, 0.95), ratio=(0.85, 1.15), antialias=True),
                          T.RandomHorizontalFlip(), T.RandomVerticalFlip(0.1), T.ColorJitter(0.4, 0.4, 0.3, 0.02)])
    test_tf = T.Compose([T.CenterCrop(int(crop / context * 1.3)), T.Resize(inp, antialias=True)])

    class DS(torch.utils.data.Dataset):
        def __init__(self, X, y, tf): self.X, self.y, self.tf = X, y, tf
        def __len__(self): return len(self.X)
        def __getitem__(self, i): return self.tf(torch.from_numpy(self.X[i]).permute(2, 0, 1)), self.y[i]

    dev = TORCH_DEVICE
    mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1) * 255
    std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1) * 255
    norm = lambda x: (x.to(dev).float() - mean) / std
    dl_tr = torch.utils.data.DataLoader(DS(Xtr, ytr, train_tf), bs, shuffle=True, drop_last=True)
    dl_va = torch.utils.data.DataLoader(DS(Xva, yva, test_tf), 512)
    model = timm.create_model("convnext_tiny.fb_in22k", pretrained=True, num_classes=4).to(dev)
    counts = np.bincount(ytr, minlength=4)
    crit = nn.CrossEntropyLoss(weight=torch.tensor((counts.max() / counts) ** 0.5, dtype=torch.float32, device=dev), label_smoothing=0.05)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-4, total_steps=epochs * len(dl_tr), pct_start=0.1)
    out.parent.mkdir(parents=True, exist_ok=True)
    for ep in range(epochs):
        t0 = time.time(); model.train()
        for x, y in dl_tr:
            loss = crit(model(norm(x)), y.to(dev)); opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            p = torch.cat([model(norm(x)).softmax(1).cpu() for x, _ in dl_va]).numpy()
        acc = [(p.argmax(1)[yva == k] == k).mean() for k in range(4)]
        print(f"epoch {ep + 1} ({time.time() - t0:.0f} sn): " + " ".join(f"{c} {a:.3f}" for c, a in zip(CLASSES, acc)))
        torch.save({"state_dict": model.state_dict(), "arch": "convnext_tiny.fb_in22k", "classes": CLASSES,
                    "crop": crop, "context": context, "input": inp}, out)
    return out


if TRAIN_CLASSIFIER:
    CLASSIFIER = train_classifier()

# B · Çıkarım

## 1 · Detektör tahminleri

In [ ]:
from ultralytics import YOLO


def _pack(xy, sc, cl):
    return (np.asarray(xy, dtype=np.float32).reshape(-1, 4), np.asarray(sc, dtype=np.float32), np.asarray(cl, dtype=np.int64))


def predict_full(model, flip=False, imgsz=1280):
    out = {}
    for k, i in enumerate(IDS, 1):
        im = Image.open(IMG_DIR / f"{i}.jpg").convert("RGB"); W = im.width
        b = model.predict(ImageOps.mirror(im) if flip else im, imgsz=imgsz, conf=CONF, max_det=MAX_DET_MODEL,
                          device=DEVICE, verbose=False)[0].boxes
        xy = b.xyxy.cpu().numpy()
        if flip:
            xy = np.stack([W - xy[:, 2], xy[:, 1], W - xy[:, 0], xy[:, 3]], 1) if len(xy) else xy
        sc, cl = b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int)
        keep = (xy[:, 2] - xy[:, 0]) * (xy[:, 3] - xy[:, 1]) >= MIN_AREA_PRED
        out[i] = _pack(xy[keep], sc[keep], cl[keep])
        if k % 500 == 0:
            print(" ", k, flush=True)
    return out


def windows(W, H, size=TILE, overlap=TILE_OVERLAP):
    def st(n):
        if n <= size:
            return [0]
        step = int(size * (1 - overlap))
        return sorted(set(list(range(0, n - size, step)) + [n - size]))
    return [(x, y, min(x + size, W), min(y + size, H)) for y in st(H) for x in st(W)]


def predict_tiles(model):
    """800 px tile'lar; tile'ın iç kenarına değen (kesilmiş) kutular atılır."""
    out = {}
    for k, i in enumerate(IDS, 1):
        im = Image.open(IMG_DIR / f"{i}.jpg").convert("RGB"); W, H = im.size
        xs, ss, cs = [], [], []
        for (x0, y0, x1, y1) in windows(W, H):
            if (x1 - x0, y1 - y0) == (W, H):
                continue
            b = model.predict(im.crop((x0, y0, x1, y1)), imgsz=TILE_IMGSZ, conf=CONF, max_det=MAX_DET_MODEL,
                              device=DEVICE, verbose=False)[0].boxes
            xy = b.xyxy.cpu().numpy(); c = b.conf.cpu().numpy(); cl = b.cls.cpu().numpy().astype(int); e = 2
            cut = (((xy[:, 0] < e) & (x0 > 0)) | ((xy[:, 1] < e) & (y0 > 0)) |
                   ((xy[:, 2] > x1 - x0 - e) & (x1 < W)) | ((xy[:, 3] > y1 - y0 - e) & (y1 < H)))
            xy, c, cl = xy[~cut], c[~cut], cl[~cut]
            xy[:, [0, 2]] += x0; xy[:, [1, 3]] += y0
            xs.append(xy); ss.append(c); cs.append(cl)
        out[i] = _pack(np.concatenate(xs) if xs else [], np.concatenate(ss) if ss else [], np.concatenate(cs) if cs else [])
        if k % 500 == 0:
            print(" ", k, flush=True)
    return out


def cached(name, fn):
    f = CACHE / f"{SPLIT}_{name}.npy"
    if f.exists():
        return np.load(f, allow_pickle=True).item()
    print("tahmin:", name, flush=True)
    r = fn(); np.save(f, r, allow_pickle=True); return r


PRED = {}
for name, w in DETECTORS.items():
    model = YOLO(str(w))
    assert [model.names[k] for k in range(4)] == CLASSES, model.names
    PRED[f"{name}_full"] = cached(f"{name}_full", lambda: predict_full(model))
    PRED[f"{name}_tiles"] = cached(f"{name}_tiles", lambda: predict_tiles(model))
    if name not in NO_TTA:
        PRED[f"{name}_flip"] = cached(f"{name}_flip", lambda: predict_full(model, flip=True))
        PRED[f"{name}_1600"] = cached(f"{name}_1600", lambda: predict_full(model, imgsz=1600))
    del model

## 2 · SAHI birleştirme ve ağırlıklı WBF
Üye sırası Kaggle'a gönderilen dosyalarla aynı.

In [ ]:
from ensemble_boxes import weighted_boxes_fusion


def nms_merge(*sets):
    """Aynı modelin tahmin kümelerini sınıf bazlı NMS ile birleştirir (en fazla MAX_DET kutu)."""
    out = {}
    for i in IDS:
        xy = np.concatenate([s[i][0] for s in sets]); sc = np.concatenate([s[i][1] for s in sets])
        cl = np.concatenate([s[i][2] for s in sets])
        if len(sc):
            keep = batched_nms(torch.tensor(xy), torch.tensor(sc), torch.tensor(cl), SAHI_IOU)[:MAX_DET].numpy()
            xy, sc, cl = xy[keep], sc[keep], cl[keep]
        out[i] = (xy, sc, cl)
    return out


def wbf(members, weights):
    out = {}
    for i in IDS:
        W, H = SIZES[i]; bl, sl, ll = [], [], []
        for m in members:
            xy, sc, cl = m[i]; o = np.argsort(-sc)[:WBF_TOP]
            bl.append(np.clip(xy[o] / [W, H, W, H], 0, 1).tolist()); sl.append(sc[o].tolist()); ll.append(cl[o].tolist())
        b, s, l = weighted_boxes_fusion(bl, sl, ll, weights=weights, iou_thr=WBF_IOU, skip_box_thr=0.0, conf_type="avg")
        o = np.argsort(-s)[:MAX_DET]
        out[i] = _pack(b[o] * [W, H, W, H], s[o], l[o].astype(int))
    return out


import warnings
warnings.filterwarnings("ignore", message="Zero area box")
core = ["yolo26m_best", "D2", "colab26m"]
members = ([nms_merge(PRED[f"{n}_full"], PRED[f"{n}_tiles"]) for n in core] +      # SAHI üyeleri
           [nms_merge(PRED[f"{n}_flip"]) for n in core] +                            # flip üyeleri
           [nms_merge(PRED[f"{n}_1600"]) for n in core])                             # 1600 px üyeleri
weights = [1] * 3 + [TTA_WEIGHT] * 6
if ENSEMBLE == "final5":
    members += [nms_merge(PRED["C_full"], PRED["C_tiles"]), nms_merge(PRED["C_flip"]), nms_merge(PRED["C_1600"]),
                nms_merge(PRED["copypaste_van_full"], PRED["copypaste_van_tiles"])]
    weights += [1, TTA_WEIGHT, TTA_WEIGHT, 1]
ENS = wbf(members, weights)
print(ENSEMBLE, "·", len(members), "üye")
print("WBF bitti:", sum(len(v[1]) for v in ENS.values()), "kutu")

## 3 · Van/car sınıflandırıcısı ile yeniden puanlama

In [ ]:
import timm
import torchvision.transforms.v2 as T

ck = torch.load(CLASSIFIER, map_location="cpu")
cls_model = timm.create_model(ck["arch"], pretrained=False, num_classes=4)
cls_model.load_state_dict(ck["state_dict"]); cls_model.to(TORCH_DEVICE).eval()
CROP, CONTEXT, INP = ck["crop"], ck["context"], ck["input"]
TEST_TF = T.Compose([T.CenterCrop(int(CROP / CONTEXT * 1.3)), T.Resize(INP, antialias=True)])
MEAN = torch.tensor([0.485, 0.456, 0.406], device=TORCH_DEVICE).view(1, 3, 1, 1) * 255
STD = torch.tensor([0.229, 0.224, 0.225], device=TORCH_DEVICE).view(1, 3, 1, 1) * 255
CAR, VAN = CID["car"], CID["van"]


@torch.no_grad()
def classify(img_id, boxes):
    im = Image.open(IMG_DIR / f"{img_id}.jpg").convert("RGB"); crops = []
    for x1, y1, x2, y2 in boxes:
        cx, cy, s = (x1 + x2) / 2, (y1 + y2) / 2, max(x2 - x1, y2 - y1) * CONTEXT
        c = np.asarray(im.crop((cx - s / 2, cy - s / 2, cx + s / 2, cy + s / 2)).resize((CROP, CROP), Image.BILINEAR))
        crops.append(TEST_TF(torch.from_numpy(c).permute(2, 0, 1)))
    probs = []
    for k in range(0, len(crops), 512):
        x = (torch.stack(crops[k:k + 512]).to(TORCH_DEVICE).float() - MEAN) / STD
        probs.append(cls_model(x).softmax(1).cpu().numpy())
    return np.concatenate(probs)


def rescore(xy, sc, cl, p, a=CLS_A):
    """q = (1-a)·onehot(detektör) + a·p (car/van üzerinde); kutu aynı sınıfla skor·q_aynı, diğer sınıfla skor·q_diğer."""
    s_same = sc.copy(); m = np.isin(cl, [CAR, VAN]) & (sc >= CLS_MIN_SCORE)
    xs, ss, cs = [xy], [s_same], [cl]
    if m.any():
        pc = p[:, [CAR, VAN]]; pc = pc / pc.sum(1, keepdims=True)
        is_van = (cl[m] == VAN).astype(int)
        q = (1 - a) * np.stack([1 - is_van, is_van], 1) + a * pc
        idx = np.arange(m.sum())
        s_same[m] = sc[m] * q[idx, is_van]
        xs.append(xy[m]); ss.append(sc[m] * q[idx, 1 - is_van]); cs.append(np.where(is_van == 1, CAR, VAN))
    xy2, s2, c2 = np.concatenate(xs), np.concatenate(ss), np.concatenate(cs)
    keep = s2 > 1e-4
    return xy2[keep], s2[keep], c2[keep]


FINAL = {}
for k, (i, (xy, sc, cl)) in enumerate(ENS.items(), 1):
    m = np.isin(cl, [CAR, VAN]) & (sc >= CLS_MIN_SCORE)
    p = classify(i, xy[m]) if m.any() else np.zeros((0, 4))
    FINAL[i] = rescore(xy, sc, cl, p)
    if k % 500 == 0:
        print(" ", k, flush=True)

## 4 · Submission (test) veya mAP@0.5 (val)

In [ ]:
def to_rows(i):
    xy, sc, cl = FINAL[i]
    w, h = xy[:, 2] - xy[:, 0], xy[:, 3] - xy[:, 1]
    o = [j for j in np.argsort(-sc) if w[j] * h[j] >= MIN_AREA_PRED][:MAX_DET]
    return [(CLASSES[cl[j]], float(sc[j]), float(xy[j, 0]), float(xy[j, 1]), float(w[j]), float(h[j])) for j in o]


if SPLIT == "test":
    rows = [(i, " ".join(f"{l} {s:.4f} {x:.1f} {y:.1f} {w:.1f} {h:.1f}" for l, s, x, y, w, h in to_rows(i)) or "none") for i in IDS]
    sub = pd.DataFrame(rows, columns=["image_id", "PredictionString"]); sub.to_csv(OUT_CSV, index=False)
    ref = pd.read_csv(DATA / "sample_submission.csv")
    assert list(sub.columns) == ["image_id", "PredictionString"] and set(sub.image_id) == set(ref.image_id)
    print(f"yazıldı: {OUT_CSV} · {len(sub)} satır · {sum(len(to_rows(i)) for i in IDS)} kutu")

In [ ]:
def map50(preds, gt_df, ids, min_area=200, iou_thr=0.5):
    """Kaggle metriğinin kopyası: sınıf bazlı AP@0.5 (VOC, tüm noktalar), 200 px² altı tahminler yok sayılır."""
    aps = {}
    for c in CLASSES:
        g = {i: d[["x", "y", "w", "h"]].to_numpy(float) for i, d in gt_df[gt_df.label == c].groupby("image_id") if i in ids}
        n_gt = sum(len(v) for v in g.values())
        dets = sorted(((i, s, np.array([x, y, w, h])) for i in ids for (l, s, x, y, w, h) in preds[i]
                       if l == c and w * h >= min_area), key=lambda d: -d[1])
        used = {i: np.zeros(len(v), bool) for i, v in g.items()}; tp = []
        for i, s, b in dets:
            t = 0
            if i in g and len(g[i]):
                G = g[i]; x1 = np.maximum(b[0], G[:, 0]); y1 = np.maximum(b[1], G[:, 1])
                x2 = np.minimum(b[0] + b[2], G[:, 0] + G[:, 2]); y2 = np.minimum(b[1] + b[3], G[:, 1] + G[:, 3])
                inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
                iou = inter / (b[2] * b[3] + G[:, 2] * G[:, 3] - inter); j = int(iou.argmax())
                if iou[j] >= iou_thr and not used[i][j]:
                    used[i][j] = True; t = 1
            tp.append(t)
        tp = np.array(tp, float); ctp = np.cumsum(tp)
        rec = ctp / max(n_gt, 1); prec = ctp / np.arange(1, len(tp) + 1)
        mrec = np.concatenate(([0], rec, [1])); mpre = np.concatenate(([0], prec, [0]))
        mpre = np.flip(np.maximum.accumulate(np.flip(mpre))); k = np.where(mrec[1:] != mrec[:-1])[0]
        aps[c] = float(np.sum((mrec[k + 1] - mrec[k]) * mpre[k + 1]))
    return float(np.mean(list(aps.values()))), aps


if SPLIT == "val":
    gt = pd.read_csv(DATA / "train" / "annotations.csv"); gt["label"] = gt.label.str.strip().str.lower()
    mp, aps = map50({i: to_rows(i) for i in IDS}, gt, set(IDS))
    print(f"val mAP@0.5 {mp:.4f} | " + " ".join(f"{c} {v:.3f}" for c, v in aps.items()))